### Install New Libraries

In [ ]:
!pip install ddgs trafilatura -q

### Setup

In [ ]:
import os
from openai import OpenAI
from dotenv import load_dotenv
import json
from pprint import pprint
from IPython.display import Markdown, display
from ddgs import DDGS
import trafilatura

load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

if OPENAI_API_KEY is None:
    raise Exception("API key is missing.")

client = OpenAI(api_key=OPENAI_API_KEY)

MODEL = "gpt-4.1-mini"

### Step 1: Define the Tools

In [ ]:
def search_web(query: str):
    # Search the web using DuckDuckGo browser. Returns 3 results.
    ddgs = DDGS()
    results = ddgs.text(query, max_results=3)
    print(f"  \u2705 Got Results\n")
    return json.dumps(results, indent=2)

In [ ]:
def fetch_url(url: str):
    # Fetch the URL content using Trafilatura. Returns the extracted text if successful, otherwise None.
    downloaded = trafilatura.fetch_url(url)
    if downloaded:
        text = trafilatura.extract(downloaded)
        if text:
            print(f"  \u2705 Got text: {len(text)} characters\n")
            return text
    print(f"  \u274C Failed to fetch or extracte text from {url}\n")
    return f"Could not fetch or extract text from {url}. Try a different source."

In [ ]:
# Test the search_web function
search_web("AI in healthcare in 2030")

In [ ]:
# Test the fetch_url function
result = fetch_url("https://en.wikipedia.org/wiki/Artificial_intelligence_in_healthcare")
print(result)

### Step 2: Describe as LLM tools

In [ ]:
tools = []

In [ ]:
search_web_function = {
    "name": "search_web",
    "description": "Search the web using DuckDuckGo browser. Returns 3 results.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "The search query"
            }
        },
        "required": ["query"]
    }
}

# Add the search_web function to the list of available tools
tools.append({"type": "function", "function": search_web_function})

In [ ]:
fetch_url_function = {
    "name": "fetch_url",
    "description": "Fetch the content of a URL using trafilatura.",
    "parameters": {
        "type": "object",
        "properties": {
            "url": {
                "type": "string",
                "description": "The URL to fetch"
            }
        },
        "required": ["url"]
    }
}

# Add to the list of available tools
tools.append({"type": "function", "function": fetch_url_function})

In [ ]:
# Check the list of available tools
for tool in tools:
    print(tool["function"]["name"])

### Step 3: Tool Call Handler

In [ ]:
def handle_tool_call(tool_calls):
    tool_results = []

    for tool_call in tool_calls:
        function_name = tool_call.function.name
        args = json.loads(tool_call.function.arguments)

        print(f"  \U0001f527 Handling tool call for function: {function_name} with arguments: {args}") # For debugging

        # Route to the appropriate function based on function_name
        if function_name == "search_web":
            # Actually perform the web search, i.e. call the tool
            result = search_web(args["query"])
            content = f"Search results: {result}"
        elif function_name == "fetch_url":
            result = fetch_url(args["url"])
            content = f"Fetched URL content: {result}"
        else:
            content = f"Unknown tool call: {function_name}"

        tool_results.append({
            "role": "tool",
            "content": content,
            "tool_call_id": tool_call.id
        })

    # Return what to add to the context about tool call results, a list of dictionaries
    return tool_results


### Step 4: The System Prompt

This tells the LLM who it is and how to behave. The key things:

- What its job is
- What tools it has
- What process to follow
- What output format to produce

In [ ]:
RESEARCH_AGENT_PROMPT = """You are a research specialist. Your job is to research a given topic
and produce a comprehensive research brief.

IMPORTANT: The word "DONE:" is a control signal, not a label. Never use it as a heading, section marker, or inline annotation.
ONLY use the word "DONE:" as per the instructions below -- it has to come at the start of a reply.

You have access to two tools:
- search_web: Search the web for information
- fetch_url: Fetch and read the full content of a web page

Your typical process:
1. Search for the topic to find relevant sources
2. Reflect on the search results — which sources look most relevant and why?
3. Fetch the full content of the 2-3 best URLs
4. Reflect on what you have gathered. Do you have enough? Are there gaps?
5. If there are gaps, search again with a different query
6. When you have enough information from at least 3 different sources, synthesize into a research brief

You MUST gather information from at least 6 distinct sources before delivering your brief.
If you have fewer than 6 sources, keep searching.

When you are ready to deliver your final research brief, start your response with "DONE:" followed by the brief itself.
It is imperative that "DONE:" should be at the start of the final response, so that it can be easly parsed and extracted.
You CANNOT and SHOULD NOT include "DONE:" in any other part of your response except at the very BEGINNING of the final research brief.

Your research brief MUST include:
- Key facts and statistics
- Main themes and arguments from the sources
- Notable data points
- Source URLs for attribution

Until you are ready, just keep working — search, fetch, think, reflect.
Do not rush. Take time to reflect between tool calls before deciding your next step.
Not every response needs a tool call — sometimes just thinking through what you have is the right move.
"""

### Step 5: The Agentic Loop

In [ ]:
def run_research_agent(topic: str, max_iterations: int = 10) -> str:
    """
    Run the research agent on a topic and return the research brief.

    Args:
        topic: The topic to research
        max_iterations: Safety limit to prevent infinite loops
    
    Returns:
        The research brief as a string
    """

    print(f"  \U0001F50D Starting research on: {topic}\n")
    print("=" * 60)

    # Initialize the conversation messages list with system prompt + research task
    messages = [
        {"role": "system", "content": RESEARCH_AGENT_PROMPT},
        {"role": "user", "content": f"Research the following topic and produce a comprehensive research brief:\n{topic}"}
    ]

    # Loop
    iteration = 0
    while iteration < max_iterations:
        iteration += 1 # (1,2,3,..,max_iterations)
        print(f"  \U0001F4D6 Iteration {iteration}\n")

        # 1. Call the LLM and get a response
        response = client.chat.completions.create(
            model=MODEL,
            messages=messages,
            tools=tools
        )

        message = response.choices[0].message
        messages.append(message)  # Append the LLM message to the conversation messages list

        # 2. Check if LLM called tools
        if message.tool_calls:
            tool_results = handle_tool_call(message.tool_calls)
            messages.extend(tool_results)
        # 3. Otherwise: no tools were called, read message content
        else:
            content = message.content # No tool calls, so there's just a normal text response from the LLM
            # Check if DONE:, then return
            if content.strip().upper().startswith("DONE:"):
                research_brief = content[len("DONE:"):].strip()
                print(f"  \u2705 Research brief complete!")
                return research_brief
            # Otherwise: not yet done
            else:
                print(f"  \U0001F4AD Agent is thinking:")
                pprint(content)
                # Loop continues to next iteration
        
        # 4. If we are entering the final iteration, force a final answer
        if (iteration == max_iterations - 1):
            print("  \u26a0 Safety Limit reached. Stopping research in next iteration")
            messages.append({"role": "user", "content": "You have reached the maximum number of iterations, please deliver your research brief now. You MUST respond with DONE: followed by your research brief."})
        
    # Fallback return
    return "Research incomplete within the maximum number of iterations."                


### Let's Run It!

In [ ]:
MODEL = "gpt-4.1-mini"
brief = run_research_agent("All the different species of rhinoceros")
display(Markdown(brief))